<a href="https://colab.research.google.com/github/rizkiismail9a/data-science-2026-unsia/blob/main/pertemuan_3_MuhamaRizkiIsmail_240401010126.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd, numpy as np
import missingno as msno # pip install missingno


df = pd.read_csv('/home/housing_dirty.csv')

# Jumlah missing per kolom
print(df.isnull().sum())

# Proporsi (%) missing per kolom
pct = (df.isnull().sum() / len(df) * 100).round(2)
print(pct[pct > 0]) # tampilkan yang > 0%

# Identifikasi baris dengan SEMUA nilai missing
all_missing = df[df.isnull().all(axis=1)]

# Identifikasi kolom yang perlu di-drop (> 40% missing)
threshold = 0.4

# Bisa pakai mean() atau sum dibagi len tanpa dikali 100 seperti di rumus propors
# Hasilnya sama
cols_drop = df.columns[df.isnull().mean() > threshold]
df.drop(columns=cols_drop, inplace=True)

# Visualisasi pola missing (perlu matplotlib)
msno.matrix(df) # matrix visualisasi
msno.heatmap(df) # korelasi ketidaklengkapan

# print(msno.heatmap(df))

In [15]:
import pandas as pd, numpy as np

file_path = '/content/drive/My Drive/Colab Notebooks/housing_dirty.csv'

df = pd.read_csv(file_path)

# print(df.isnull().sum())
# print(df.info())
# print(df.describe().round(2))

# Lihat data terduplikasi
n_dup = df.duplicated().sum()
# print(f'{n_dup} baris duplikat dari {len(df)} total')

# Tampilkan baris yang duplikat (semua kemunculan)
df_dup = df[df.duplicated(keep=False)]
# print(df_dup)
df.drop_duplicates(inplace=True) # 0 baris terduplikasi

# Normalisasi String
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

# Imputasi nilai kosong
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

# Tangani outlier
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
  Q1, Q3 = df[col].quantile([0.25, 0.75])
  IQR = Q3 - Q1
  # Semua nilai yang lebih kecil dari Q1 - 1.5*IQR akan "dipotong" atau diganti dengan nilai Q1 - 1.5*IQR itu sendiri. Batas ini dikenal sebagai batas bawah untuk outlier.
  df[col] = df[col].clip(lower=Q1 - 1.5*IQR, upper=Q3 + 1.5*IQR)

# Validaso
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'
print('Shape akhir:', df.shape)

df.to_csv('/content/drive/My Drive/Colab Notebooks/housing_clean.csv', index=False)

Shape akhir: (130, 7)


In [22]:
import requests, pandas as pd
from pandas import json_normalize

URL = "https://jsonplaceholder.typicode.com/users"
response = requests.get(URL, timeout=10)


if response.status_code == 200:
  data = response.json()
  df = json_normalize(data, sep='_')
  print(df[['id','name','email','address_city']])
else:
  print(f'Error: {response.status_code}')


params = {'userId': 1} # filter by user
posts = requests.get(URL, params=params).json()
df_posts = json_normalize(posts)
print(df_posts)

   id                      name          username                      email  \
0   1             Leanne Graham              Bret          Sincere@april.biz   
1   2              Ervin Howell         Antonette          Shanna@melissa.tv   
2   3          Clementine Bauch          Samantha         Nathan@yesenia.net   
3   4          Patricia Lebsack          Karianne  Julianne.OConner@kory.org   
4   5          Chelsey Dietrich            Kamren   Lucio_Hettinger@annie.ca   
5   6      Mrs. Dennis Schulist  Leopoldo_Corkery    Karley_Dach@jasper.info   
6   7           Kurtis Weissnat      Elwyn.Skiles     Telly.Hoeger@billy.biz   
7   8  Nicholas Runolfsdottir V     Maxime_Nienow       Sherwood@rosamond.me   
8   9           Glenna Reichert          Delphine    Chaim_McDermott@dana.io   
9  10        Clementina DuBuque    Moriah.Stanton     Rey.Padberg@karina.biz   

                   phone        website     address.street address.suite  \
0  1-770-736-8031 x56442  hildegard.org    